# High-throughput QC for Cellpose cell/nuclei segmentations

Use this notebook **after** running `NEW_cellpose-prep.ipynb` and Cellpose inference.

This notebook is designed for **macro-level QC** across many images:
- Did segmentation generally work?
- Which images likely failed badly?
- Which datasets need re-tuning?

Outputs include:
- per-image QC table (`PASS` / `WARN` / `FAIL`)
- per-dataset summary table
- quick visual contact sheets for fast review

## Step 1: Imports

### ▶ Run this cell (no user input required)

In [1]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import json

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
import tifffile

from infer_subc.core.file_io import list_image_files, read_czi_image

## Step 2: User inputs

### 🛑 Edit the next cell (User Input Required)

Definitions of all user-defined variables (with math details):

**Dataset and file discovery**
- `DATASET_FOLDERS`: list of dataset folders to evaluate (each should contain raw images and the workflow subfolder).
- `WORKFLOW_FOLDER_NAME`: workflow subfolder name inside each dataset folder (for example `20261112_batch-test`).
- `RAW_IMAGE_EXTENSIONS`: raw image file suffixes to search when matching masks to source images.
- `EXPECT_NUCLEI`: set `True` if nuclei segmentation is expected and should be QC-checked; set `False` for cell-only workflows.

**How the QC metrics are computed (per image)**
- `cell_count`: number of unique non-zero labels in the cell mask.
- `nuc_count`: number of unique non-zero labels in the nuclei mask.
- `cell_coverage`: fraction of voxels/pixels labeled as cell, computed as
  $cell\_coverage = \frac{\#(cell>0)}{N}$ where $N$ is total voxel/pixel count.
- `nuc_coverage`: fraction labeled as nuclei,
  $nuc\_coverage = \frac{\#(nuc>0)}{N}$.
- `nuc_outside_cell_fraction`: fraction of nuclei voxels that lie outside the cell mask,
  $nuc\_outside = \frac{\#((nuc>0) \land (cell=0))}{\#(nuc>0)}$.
- `nuc_per_cell_ratio`: object-count ratio,
  $nuc\_per\_cell = \frac{nuc\_count}{cell\_count}$ (undefined if `cell_count = 0`, treated as NaN).

**Hard FAIL thresholds**
- `MIN_CELL_COUNT`: FAIL if `cell_count < MIN_CELL_COUNT`.
- `MIN_CELL_COVERAGE`: FAIL if `cell_coverage < MIN_CELL_COVERAGE`.
- `MAX_CELL_COVERAGE`: FAIL if `cell_coverage > MAX_CELL_COVERAGE`.
- `MIN_NUC_COUNT`: FAIL if `nuc_count < MIN_NUC_COUNT` (only when `EXPECT_NUCLEI=True`).
- `MIN_NUC_COVERAGE`: FAIL if `nuc_coverage < MIN_NUC_COVERAGE` (only when nuclei are expected).
- `MAX_NUC_COVERAGE`: FAIL if `nuc_coverage > MAX_NUC_COVERAGE` (only when nuclei are expected).

**Soft WARN thresholds**
- `MAX_NUC_OUTSIDE_CELL_FRAC_WARN`: WARN if `nuc_outside_cell_fraction > MAX_NUC_OUTSIDE_CELL_FRAC_WARN`.
- `MIN_NUC_PER_CELL_WARN`: WARN if `nuc_per_cell_ratio < MIN_NUC_PER_CELL_WARN`.
- `MAX_NUC_PER_CELL_WARN`: WARN if `nuc_per_cell_ratio > MAX_NUC_PER_CELL_WARN`.

**Status logic**
1. Start as `PASS`.
2. If any FAIL rule is triggered, status becomes `FAIL`.
3. Else, if any WARN rule is triggered, status becomes `WARN`.

**Output and visualization**
- `OUTPUT_REPORT_DIRNAME`: folder name created inside each workflow folder for QC outputs.
- `INCLUDE_ALL_IMAGES_IN_CONTACT_SHEET`: if `True`, include every image in each dataset contact sheet.
- `SAMPLE_CONTACT_SHEET_PER_DATASET`: number of images to include only when `INCLUDE_ALL_IMAGES_IN_CONTACT_SHEET=False`.
- `BACKGROUND_SOURCE`: background image source for overlays. Use `"prepared"` (default; from `cell_input/` then `nuclei_input/`), `"raw"` (raw image), or `"auto"` (prepared first, then raw fallback).
- `PREFERRED_DISPLAY_CHANNEL`: optional raw channel index for raw-background mode (`None` uses max projection across channels).

**Practical interpretation notes**
- Coverage thresholds are unitless fractions in `[0, 1]`.
- Example: `MIN_CELL_COVERAGE = 0.0005` means at least 0.05% of voxels/pixels must be labeled as cell.
- For sparse 3D datasets, coverage values can be very small; tune thresholds using one pilot run and inspect the QC table distribution.

In [2]:
### USER INPUT REQUIRED ###
DATASET_FOLDERS = [
    Path("/users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1"),
    # Path("/users/s/r/srhoads/MJFF_Cellpose-batch-testing/000001"),
]

WORKFLOW_FOLDER_NAME = "20261112_batch-test"
RAW_IMAGE_EXTENSIONS = [".tiff", ".tif", ".czi"]
EXPECT_NUCLEI = False

# Hard fail thresholds
MIN_CELL_COUNT = 1
MIN_CELL_COVERAGE = 0.0005
MAX_CELL_COVERAGE = 0.95
MIN_NUC_COUNT = 1
MIN_NUC_COVERAGE = 0.0001
MAX_NUC_COVERAGE = 0.70

# Soft warning thresholds
MAX_NUC_OUTSIDE_CELL_FRAC_WARN = 0.20
MIN_NUC_PER_CELL_WARN = 1
MAX_NUC_PER_CELL_WARN = 1

OUTPUT_REPORT_DIRNAME = "qc_reports"
INCLUDE_ALL_IMAGES_IN_CONTACT_SHEET = True
SAMPLE_CONTACT_SHEET_PER_DATASET = 24
BACKGROUND_SOURCE = "prepared"  # "prepared", "raw", or "auto"
PREFERRED_DISPLAY_CHANNEL = None  # raw mode only; e.g. 7

## Step 3: Build manifest

### ▶ Run this cell (no user input required)

This finds dataset images, mask files, and expected workflow locations.

In [3]:
def _label_count(arr: np.ndarray) -> int:
    if arr is None:
        return 0
    u = np.unique(arr)
    return int(np.sum(u > 0))

# Convert a mask to ZYX format (if needed); 
def _to_zyx_mask(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr)
    # if arr.ndim == 2, add a new axis for Z
    if arr.ndim == 2:
        return arr[None, ...]
    # if arr.ndim == 3, it's already in ZYX format
    if arr.ndim == 3:
        return arr
    # if arr.ndim == 4, assume it's in CZYX format and take the first channel
    if arr.ndim >= 4:
        # Use first channel/frame if extra dims exist
        return np.asarray(arr[0])
    return arr


def _find_raw_files(dataset_folder: Path, exts: list[str]) -> dict[str, Path]:
    m = {}
    for ext in exts:
        for p in list_image_files(dataset_folder, ext):
            m[Path(p).stem] = Path(p)
    return m


records = []
for dataset_folder in DATASET_FOLDERS:
    dataset_folder = Path(dataset_folder)
    if not dataset_folder.exists():
        print(f"WARNING: dataset not found: {dataset_folder}")
        continue

    workflow_root = dataset_folder / WORKFLOW_FOLDER_NAME
    cell_mask_dir = workflow_root / "cell_masks"
    nuclei_mask_dir = workflow_root / "nuclei_masks"

    if not cell_mask_dir.exists():
        print(f"WARNING: missing cell mask folder: {cell_mask_dir}")
        continue

    raw_map = _find_raw_files(dataset_folder, RAW_IMAGE_EXTENSIONS)

    # Cellpose output default is *_cp_masks.tif
    cell_files = sorted(cell_mask_dir.glob("*_cp_masks.tif"))
    if not cell_files:
        # fallback
        cell_files = sorted(cell_mask_dir.glob("*.tif"))

    for cf in cell_files:
        stem = cf.name.replace("_cp_masks.tif", "") if cf.name.endswith("_cp_masks.tif") else cf.stem
        nf = nuclei_mask_dir / cf.name
        raw_file = raw_map.get(stem, None)

        records.append({
            "dataset_id": dataset_folder.name,
            "dataset_folder": str(dataset_folder),
            "workflow_root": str(workflow_root),
            "image_stem": stem,
            "raw_file": str(raw_file) if raw_file is not None else None,
            "cell_mask_file": str(cf),
            "nuclei_mask_file": str(nf) if nf.exists() else None,
        })

manifest = pd.DataFrame(records)
if manifest.empty:
    raise RuntimeError("No mask files were found. Check DATASET_FOLDERS and WORKFLOW_FOLDER_NAME.")

print(f"Found {len(manifest)} images across {manifest['dataset_id'].nunique()} dataset(s).")
manifest.head(10)

Found 10 images across 1 dataset(s).


,dataset_id,dataset_folder,workflow_root,image_stem,raw_file,cell_mask_file,nuclei_mask_file
0,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None
1,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None
2,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None
3,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None
4,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None
5,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None
6,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None
7,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None
8,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None
9,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None


## Step 4: Compute macro QC metrics

### ▶ Run this cell (no user input required)

This computes object counts, area fractions, and simple FAIL/WARN rules per image.

In [4]:
def _coverage_fraction(lbl_zyx: np.ndarray | None) -> float:
    if lbl_zyx is None:
        return np.nan
    fg = lbl_zyx > 0
    return float(fg.mean())


def _outside_fraction(nuc_zyx: np.ndarray | None, cell_zyx: np.ndarray | None) -> float:
    if nuc_zyx is None or cell_zyx is None:
        return np.nan
    nuc_fg = nuc_zyx > 0
    if nuc_fg.sum() == 0:
        return 0.0
    outside = nuc_fg & ~(cell_zyx > 0)
    return float(outside.sum() / nuc_fg.sum())


qc_rows = []
for r in manifest.to_dict(orient="records"):
    cell = _to_zyx_mask(tifffile.imread(r["cell_mask_file"])) if r["cell_mask_file"] else None
    nuc = _to_zyx_mask(tifffile.imread(r["nuclei_mask_file"])) if r["nuclei_mask_file"] else None

    cell_count = _label_count(cell)
    nuc_count = _label_count(nuc)
    cell_cov = _coverage_fraction(cell)
    nuc_cov = _coverage_fraction(nuc)
    nuc_outside = _outside_fraction(nuc, cell)
    nuc_per_cell = (nuc_count / cell_count) if cell_count > 0 else np.nan

    reasons_fail = []
    reasons_warn = []

    if cell_count < MIN_CELL_COUNT:
        reasons_fail.append("cell_count_too_low")
    if np.isfinite(cell_cov) and cell_cov < MIN_CELL_COVERAGE:
        reasons_fail.append("cell_coverage_too_low")
    if np.isfinite(cell_cov) and cell_cov > MAX_CELL_COVERAGE:
        reasons_fail.append("cell_coverage_too_high")

    if EXPECT_NUCLEI:
        if r["nuclei_mask_file"] is None:
            reasons_fail.append("missing_nuclei_mask_file")
        if nuc_count < MIN_NUC_COUNT:
            reasons_fail.append("nuc_count_too_low")
        if np.isfinite(nuc_cov) and nuc_cov < MIN_NUC_COVERAGE:
            reasons_fail.append("nuc_coverage_too_low")
        if np.isfinite(nuc_cov) and nuc_cov > MAX_NUC_COVERAGE:
            reasons_fail.append("nuc_coverage_too_high")

        if np.isfinite(nuc_outside) and nuc_outside > MAX_NUC_OUTSIDE_CELL_FRAC_WARN:
            reasons_warn.append("many_nuclei_outside_cells")

        if np.isfinite(nuc_per_cell) and nuc_per_cell < MIN_NUC_PER_CELL_WARN:
            reasons_warn.append("low_nuc_per_cell_ratio")
        if np.isfinite(nuc_per_cell) and nuc_per_cell > MAX_NUC_PER_CELL_WARN:
            reasons_warn.append("high_nuc_per_cell_ratio")

    status = "PASS"
    if reasons_fail:
        status = "FAIL"
    elif reasons_warn:
        status = "WARN"

    qc_rows.append({
        **r,
        "cell_count": int(cell_count),
        "nuc_count": int(nuc_count),
        "cell_coverage": float(cell_cov) if np.isfinite(cell_cov) else np.nan,
        "nuc_coverage": float(nuc_cov) if np.isfinite(nuc_cov) else np.nan,
        "nuc_outside_cell_fraction": float(nuc_outside) if np.isfinite(nuc_outside) else np.nan,
        "nuc_per_cell_ratio": float(nuc_per_cell) if np.isfinite(nuc_per_cell) else np.nan,
        "qc_status": status,
        "fail_reasons": ";".join(reasons_fail),
        "warn_reasons": ";".join(reasons_warn),
    })

qc_table = pd.DataFrame(qc_rows)

summary = (
    qc_table.groupby(["dataset_id", "qc_status"]).size().rename("n_images").reset_index()
    .sort_values(["dataset_id", "qc_status"])
)

print("QC status counts by dataset:")
display(summary)

qc_table.head(20)

QC status counts by dataset:


,dataset_id,qc_status,n_images
0,KK-1,PASS,10


,dataset_id,dataset_folder,workflow_root,image_stem,raw_file,cell_mask_file,nuclei_mask_file,cell_count,nuc_count,cell_coverage,nuc_coverage,nuc_outside_cell_fraction,nuc_per_cell_ratio,qc_status,fail_reasons,warn_reasons
0,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None,1,0,0.185382,NaN,NaN,0.0,PASS,,
1,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None,8,0,0.182766,NaN,NaN,0.0,PASS,,
2,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None,6,0,0.158139,NaN,NaN,0.0,PASS,,
3,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None,2,0,0.168802,NaN,NaN,0.0,PASS,,
4,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None,2,0,0.193878,NaN,NaN,0.0,PASS,,
5,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None,4,0,0.155433,NaN,NaN,0.0,PASS,,
6,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None,5,0,0.163242,NaN,NaN,0.0,PASS,,
7,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None,6,0,0.093462,NaN,NaN,0.0,PASS,,
8,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None,4,0,0.208864,NaN,NaN,0.0,PASS,,
9,KK-1,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,04102026_iAstrocytesD21_BFP2EEA1_125ng_mTurLAM...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,/users/s/r/srhoads/MJFF_Cellpose-batch-testing...,None,6,0,0.229193,NaN,NaN,0.0,PASS,,


## Step 5: Save QC tables and plots

### ▶ Run this cell (no user input required)

In [5]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
saved_outputs = []

for dataset_id, sub in qc_table.groupby("dataset_id"):
    sub = sub.copy().sort_values("image_stem").reset_index(drop=True)
    sub["image_qc_id"] = [f"IMG_{i+1:04d}" for i in range(len(sub))]

    workflow_root = Path(sub["workflow_root"].iloc[0])
    out_dir = workflow_root / OUTPUT_REPORT_DIRNAME
    out_dir.mkdir(parents=True, exist_ok=True)

    out_csv = out_dir / f"cellpose_qc_table_{timestamp}.csv"
    out_json = out_dir / f"cellpose_qc_summary_{timestamp}.json"

    # Save full image names plus short IDs used in contact sheets.
    sub.sort_values(["qc_status", "image_qc_id"]).to_csv(out_csv, index=False)

    status_counts = sub["qc_status"].value_counts().to_dict()
    out_json.write_text(json.dumps({
        "dataset_id": dataset_id,
        "n_images": int(len(sub)),
        "status_counts": status_counts,
        "timestamp": timestamp,
    }, indent=2))

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
    axes[0].hist(sub["cell_count"], bins=25)
    axes[0].set_title("Cell count")

    axes[1].hist(sub["nuc_count"], bins=25)
    axes[1].set_title("Nuclei count")

    axes[2].hist(sub["cell_coverage"].dropna(), bins=25)
    axes[2].set_title("Cell coverage fraction")

    fig.suptitle(f"Macro QC distributions: {dataset_id}")
    fig.tight_layout()

    out_png = out_dir / f"cellpose_qc_distributions_{timestamp}.png"
    fig.savefig(out_png, dpi=160)
    plt.close(fig)

    saved_outputs.extend([out_csv, out_json, out_png])

print("Saved QC outputs:")
for p in saved_outputs:
    print(f"- {p}")

Saved QC outputs:
- /users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test/qc_reports/cellpose_qc_table_20261116_155312.csv
- /users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test/qc_reports/cellpose_qc_summary_20261116_155312.json
- /users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test/qc_reports/cellpose_qc_distributions_20261116_155312.png


## Step 6: Build contact sheets (fast visual triage)

### ▶ Run this cell (no user input required)

This saves overview PNG contact sheets, prioritizing `FAIL` and `WARN` images first.
By default overlays are drawn on prepared inputs (`cell_input`/`nuclei_input`) instead of raw images.
Each panel uses an image QC ID (for example `IMG_0007`), and the full filename is retained in the CSV mapping.

In [6]:
import glasbey

rng = np.random.default_rng(7)
sheet_outputs = []

def _project_for_display(arr: np.ndarray) -> np.ndarray:
    a = np.asarray(arr)
    if a.ndim == 2:
        return a
    if a.ndim == 3:
        return a.max(axis=0)
    if a.ndim == 4:
        # handle likely ZCYX or CZYX prepared/raw arrays
        if a.shape[1] <= 4 and a.shape[1] < a.shape[0]:
            # ZCYX -> take first channel
            return a[:, 0, :, :].max(axis=0)
        if a.shape[0] <= 4:
            # CZYX -> take first channel
            return a[0].max(axis=0)
        return a[0].max(axis=0)
    # fallback
    return np.squeeze(a)

def _load_prepared_background_MAX(row: dict) -> np.ndarray | None:
    workflow_root = Path(row["workflow_root"])
    stem = row["image_stem"]
    candidates = [
        workflow_root / "cell_input" / f"{stem}.tif",
        workflow_root / "nuclei_input" / f"{stem}.tif",
    ]
    for p in candidates:
        if p.exists():
            try:
                return _project_for_display(tifffile.imread(p))
            except Exception:
                continue
    return None

def _load_raw_background_MAX(row: dict) -> np.ndarray | None:
    raw_path = row.get("raw_file")
    if raw_path is None or not Path(raw_path).exists():
        return None
    try:
        raw, _meta = read_czi_image(Path(raw_path))
        raw = np.asarray(raw)
        if raw.ndim == 4:
            # CZYX expected from reader
            if PREFERRED_DISPLAY_CHANNEL is not None and int(PREFERRED_DISPLAY_CHANNEL) < raw.shape[0]:
                return raw[int(PREFERRED_DISPLAY_CHANNEL)].max(axis=0)
            return raw.max(axis=0).max(axis=0)
        if raw.ndim == 3:
            return raw.max(axis=0)
        if raw.ndim == 2:
            return raw
    except Exception:
        return None
    return None


def _load_prepared_background(row: dict) -> np.ndarray | None:
    workflow_root = Path(row["workflow_root"])
    stem = row["image_stem"]
    candidates = [
        workflow_root / "cell_input" / f"{stem}.tif",
        workflow_root / "nuclei_input" / f"{stem}.tif",
    ]
    for p in candidates:
        if p.exists():
            try:
                return tifffile.imread(p)
            except Exception:
                continue
    return None

def _load_raw_background(row: dict) -> np.ndarray | None:
    raw_path = row.get("raw_file")
    if raw_path is None or not Path(raw_path).exists():
        return None
    try:
        raw, _meta = read_czi_image(Path(raw_path))
        raw = np.asarray(raw)
        if raw.ndim == 4:
            # CZYX expected from reader
            if PREFERRED_DISPLAY_CHANNEL is not None and int(PREFERRED_DISPLAY_CHANNEL) < raw.shape[0]:
                return raw[int(PREFERRED_DISPLAY_CHANNEL)]
            return raw
        if raw.ndim == 3:
            return raw
        if raw.ndim == 2:
            return raw
    except Exception:
        return None
    return None

def _random_label_cmap(max_label: int, seed: int = 0) -> ListedColormap:
    """
    Generate a random label colormap with Glasbey colors. The first color (label 0) is set to black, and subsequent labels are assigned random colors.
    """
    palette = glasbey.create_palette(palette_size=256, 
                                     lightness_bounds=(55, 95),
                                     chroma_bounds=(40, 100),
                                     colorblind_safe=False)  # Set True if needed
    rng_local = np.random.default_rng(seed)
    rng_local.shuffle(palette)  # Shuffle the palette to get random colors
    return ListedColormap(palette) # make it a color map



for dataset_id, sub in qc_table.groupby("dataset_id"):
    sub = sub.copy().sort_values("image_stem").reset_index(drop=True)
    sub["image_qc_id"] = [f"IMG_{i+1:04d}" for i in range(len(sub))]

    workflow_root = Path(sub["workflow_root"].iloc[0])
    out_dir = workflow_root / OUTPUT_REPORT_DIRNAME
    out_dir.mkdir(parents=True, exist_ok=True)

    if INCLUDE_ALL_IMAGES_IN_CONTACT_SHEET:
        status_order = {"FAIL": 0, "WARN": 1, "PASS": 2}
        panel = sub.assign(_ord=sub["qc_status"].map(status_order).fillna(99)).sort_values(["_ord", "image_stem"]).drop(columns=["_ord"])
    else:
        fail_warn = sub[sub["qc_status"].isin(["FAIL", "WARN"])].copy()
        passed = sub[sub["qc_status"] == "PASS"].copy()

        n_total = min(SAMPLE_CONTACT_SHEET_PER_DATASET, len(sub))
        n_fw = min(len(fail_warn), n_total)
        n_pass = n_total - n_fw

        chosen = []
        if n_fw > 0:
            chosen.append(fail_warn.head(n_fw))
        if n_pass > 0 and len(passed) > 0:
            idx = rng.choice(passed.index.to_numpy(), size=min(n_pass, len(passed)), replace=False)
            chosen.append(passed.loc[idx])

        if not chosen:
            continue

        panel = pd.concat(chosen, ignore_index=True).sort_values(["qc_status", "image_stem"])

    n = len(panel)
    ncols = 4
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.6 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax in axes:
        ax.axis("off")

    for i, row in enumerate(panel.to_dict(orient="records")):
        ax = axes[i]
        ax.axis("off")

        cell = _to_zyx_mask(tifffile.imread(row["cell_mask_file"]))
        nuc = _to_zyx_mask(tifffile.imread(row["nuclei_mask_file"])) if row["nuclei_mask_file"] else None

        base_img = None
        if BACKGROUND_SOURCE == "prepared":
            base_img = _load_prepared_background_MAX(row)
        elif BACKGROUND_SOURCE == "raw":
            base_img = _load_raw_background_MAX(row)
        else:  # auto
            base_img = _load_prepared_background_MAX(row)
            if base_img is None:
                base_img = _load_raw_background_MAX(row)

        if base_img is None:
            base_img = (cell > 0).max(axis=0).astype(float)

        # create 2D label images for overlay; use max projection along Z
        cell_lbl_2d = cell.max(axis=0).astype(np.int32)
        nuc_lbl_2d = nuc.max(axis=0).astype(np.int32) if nuc is not None else np.zeros_like(cell_lbl_2d, dtype=np.int32)

        ax.imshow(base_img, cmap="gray")

        # Overlay cell and nuclei labels with transparency
        if int(cell_lbl_2d.max()) > 0:
            # anywhere cell label is > 0, mask it out for display
            cell_masked = np.ma.masked_where(cell_lbl_2d <= 0, cell_lbl_2d)
            ax.imshow(
                cell_masked,
                cmap=_random_label_cmap(int(cell_lbl_2d.max()), seed=12),
                alpha=0.45,
                interpolation="nearest",
            )

        if int(nuc_lbl_2d.max()) > 0:
            nuc_masked = np.ma.masked_where(nuc_lbl_2d <= 0, nuc_lbl_2d)
            ax.imshow(
                nuc_masked,
                cmap=_random_label_cmap(int(nuc_lbl_2d.max()), seed=29),
                alpha=0.40,
                interpolation="nearest",
            )

        short_reason = row["fail_reasons"] or row["warn_reasons"] or "ok"
        short_reason = short_reason[:45] + ("..." if len(short_reason) > 45 else "")
        ax.set_title(f"{row['qc_status']} | {row['image_qc_id']}\n{short_reason}", fontsize=8)

    fig.suptitle(f"Cellpose QC contact sheet: {dataset_id}", fontsize=12)
    fig.tight_layout()

    out_png = out_dir / f"cellpose_qc_contact_sheet_{timestamp}.png"
    fig.savefig(out_png, dpi=160)
    plt.close(fig)

    sheet_outputs.append(out_png)

print("Saved contact sheets:")
for p in sheet_outputs:
    print(f"- {p}")

Saved contact sheets:
- /users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test/qc_reports/cellpose_qc_contact_sheet_20261116_155312.png


In [14]:
# loop through each image in the qc_table and generate a contact sheet for each dataset, displaying the cell and nuclei masks overlaid on the base image. 
# The contact sheets are saved as PNG files in the specified output directory.
import scipy

aniso_zoom = 10
sheet_outputs = {}


for dataset_id, sub in qc_table.groupby("dataset_id"):
    sub = sub.copy().sort_values("image_stem").reset_index(drop=True)
    sub["image_qc_id"] = [f"IMG_{i+1:04d}" for i in range(len(sub))]

    workflow_root = Path(sub["workflow_root"].iloc[0])
    out_dir = workflow_root / OUTPUT_REPORT_DIRNAME
    out_dir.mkdir(parents=True, exist_ok=True)

    # Determine which images to include in the contact sheet based on the INCLUDE_ALL_IMAGES_IN_CONTACT_SHEET flag and the number of images to sample per dataset.
    if INCLUDE_ALL_IMAGES_IN_CONTACT_SHEET:
        status_order = {"FAIL": 0, "WARN": 1, "PASS": 2}
        panel = sub.assign(_ord=sub["qc_status"].map(status_order).fillna(99)).sort_values(["_ord", "image_stem"]).drop(columns=["_ord"])
    else:
        fail_warn = sub[sub["qc_status"].isin(["FAIL", "WARN"])].copy()
        passed = sub[sub["qc_status"] == "PASS"].copy()

        n_total = min(SAMPLE_CONTACT_SHEET_PER_DATASET, len(sub))
        n_fw = min(len(fail_warn), n_total)
        n_pass = n_total - n_fw

        chosen = []
        if n_fw > 0:
            chosen.append(fail_warn.head(n_fw))
        if n_pass > 0 and len(passed) > 0:
            idx = rng.choice(passed.index.to_numpy(), size=min(n_pass, len(passed)), replace=False)
            chosen.append(passed.loc[idx])

        if not chosen:
            continue

        panel = pd.concat(chosen, ignore_index=True).sort_values(["qc_status", "image_stem"])


    n = len(panel)
    ncols = 4
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.6 * nrows))
    axes = np.atleast_1d(axes).ravel()

    # create a contact sheet by iterating through the selected images and overlaying the cell and nuclei masks on the base image. 
    # Each subplot is labeled with the QC status, image ID, and any fail/warn reasons.
    for ax in axes:
        ax.axis("off")

    for i, row in enumerate(panel.to_dict(orient="records")):
        ax = axes[i]
        ax.axis("off")

        # Load the cell and nuclei masks for the current image
        cell = _to_zyx_mask(tifffile.imread(row["cell_mask_file"]))
        nuc = _to_zyx_mask(tifffile.imread(row["nuclei_mask_file"])) if row["nuclei_mask_file"] else None

        # Load the base image for overlay, either from prepared background, raw image, or a fallback if neither is available.
        # The base image is used as the background for displaying the cell and nuclei masks.
        # 
        base_img = None
        if BACKGROUND_SOURCE == "prepared":
            base_img = _load_prepared_background(row)
        elif BACKGROUND_SOURCE == "raw":
            base_img = _load_raw_background(row)
        else:  # auto
            base_img = _load_prepared_background(row)
            if base_img is None:
                base_img = _load_raw_background(row)


        # Create MAX intensity projections along all axes
        XY_mips = {
            "base": base_img.max(axis=0) if base_img is not None else np.zeros_like(cell.max(axis=0)),
            "cell": cell.max(axis=0),
            "nuc": nuc.max(axis=0) if nuc is not None else np.zeros_like(cell.max(axis=0))
        }

        ZY_mips = {
            "base": scipy.ndimage.zoom((base_img.max(axis=2).T), zoom=(1, aniso_zoom), order=1) if base_img is not None else scipy.ndimage.zoom(np.zeros_like(cell.max(axis=2).T), zoom=(1, aniso_zoom), order=1),
            "cell": scipy.ndimage.zoom((cell.max(axis=2).T), zoom=(1, aniso_zoom), order=1),
            "nuc": scipy.ndimage.zoom((nuc.max(axis=2).T), zoom=(1, aniso_zoom), order=1) if nuc is not None else scipy.ndimage.zoom(np.zeros_like(cell.max(axis=2).T), zoom=(1, aniso_zoom), order=1)
        }

        ZX_mips = {
            "base": scipy.ndimage.zoom((base_img.max(axis=1)), zoom=(aniso_zoom, 1), order=1) if base_img is not None else scipy.ndimage.zoom(np.zeros_like(cell.max(axis=1)), zoom=(aniso_zoom, 1), order=1),
            "cell": scipy.ndimage.zoom((cell.max(axis=1)), zoom=(aniso_zoom, 1), order=1),
            "nuc": scipy.ndimage.zoom((nuc.max(axis=1)), zoom=(aniso_zoom, 1), order=1) if nuc is not None else scipy.ndimage.zoom(np.zeros_like(cell.max(axis=1)), zoom=(aniso_zoom, 1), order=1)
        }

        # array shape --> canvas shape
        X_vox = cell.shape[2]
        Y_vox = cell.shape[1]
        Z_vox = cell.shape[0]*aniso_zoom  # Z dimension scaled by anisotropy factor

        gap = 5  # Pixel gap between views
        canvas_h = Y_vox + gap + Z_vox
        canvas_w = X_vox + gap + Z_vox

        canvases = {}
        for name in ["base", "cell", "nuc"]:
            canvas = np.full((canvas_h, canvas_w), np.nan)
            canvas[0:Y_vox, 0:X_vox] = XY_mips[name]
            canvas[0:Y_vox, X_vox + gap : X_vox + gap + Z_vox] = ZY_mips[name]
            canvas[Y_vox + gap : Y_vox + gap + Z_vox, 0:X_vox] = ZX_mips[name]
            canvases[name] = canvas


        ax.imshow(canvases["base"], cmap="gray")

        # Overlay cell and nuclei labels with transparency
        if int(XY_mips["cell"].max()) > 0:
            # anywhere cell label is > 0, mask it out for display
            cell_masked = np.ma.masked_where(canvases["cell"] <= 0, canvases["cell"])
            ax.imshow(
                cell_masked,
                cmap=_random_label_cmap(int(XY_mips["cell"].max()), seed=12),
                alpha=0.45,
                interpolation="nearest",
            )

        if int(XY_mips["nuc"].max()) > 0:
            nuc_masked = np.ma.masked_where(canvases["nuc"] <= 0, canvases["nuc"])
            ax.imshow(
                nuc_masked,
                cmap=_random_label_cmap(int(XY_mips["nuc"].max()), seed=29),
                alpha=0.40,
                interpolation="nearest",
            )

        short_reason = row["fail_reasons"] or row["warn_reasons"] or "ok"
        short_reason = short_reason[:45] + ("..." if len(short_reason) > 45 else "")
        ax.set_title(f"{row['qc_status']} | {row['image_qc_id']}\n{short_reason}", fontsize=8)

    fig.suptitle(f"Cellpose QC contact sheet: {dataset_id}", fontsize=12)
    fig.tight_layout()

    out_png = out_dir / f"cellpose_qc_contact_sheet_{timestamp}.png"
    fig.savefig(out_png, dpi=160)
    plt.close(fig)

    sheet_outputs[dataset_id] = out_png

print("Saved contact sheets:")
for p, path in sheet_outputs.items():
    print(f"- {p}: {path}")

Saved contact sheets:
- KK-1: /users/s/r/srhoads/MJFF_Cellpose-batch-testing/KK-1/20261112_batch-test/qc_reports/cellpose_qc_contact_sheet_20261116_155312.png
